# Amazon ML Challenge 2026 - Run 001: LightGBM Baseline

- **Run ID:** `run001_lgbm_tfidf`  
- **Description:** LightGBM on TF-IDF word+char n-grams plus regex catalog features, log1p target, L1 objective. Text only, no images.  
- **Expected Metric:** ~50–52% holdout SMAPE  
- **Date:** 2026-08-04  

In [ ]:
import os
import sys
import glob
from pathlib import Path

# Locate uploaded dataset containing src/amlc
candidate_paths = []
patterns = [
    "/kaggle/input/*/src/amlc",
    "/kaggle/input/*/*/src/amlc",
    "/kaggle/input/*/*/*/src/amlc",
    "/kaggle/input/datasets/*/*/src/amlc"
]
for pattern in patterns:
    for match in glob.glob(pattern):
        candidate_paths.append(str(Path(match).parents[1].resolve()))

# Remove duplicates while preserving order
candidate_paths = list(dict.fromkeys(candidate_paths))

assert len(candidate_paths) > 0, "Could not locate repository root containing src/amlc in /kaggle/input!"
REPO_PATH = Path(candidate_paths[0]).resolve()
sys.path.insert(0, str(REPO_PATH / "src"))

print(f"[OK] Found repository root at: {REPO_PATH}")
import amlc
print(f"[OK] Successfully imported amlc v{amlc.__version__}")

In [ ]:
# Ensure required lightweight packages are installed
try:
    import lightgbm
    import yaml
except ImportError:
    !pip install -q lightgbm pyyaml

In [ ]:
# Sanity check metric implementation
from amlc.metrics.smape import smape

val = smape([100.0, 200.0], [100.0, 200.0])
assert val == 0.0, f"SMAPE sanity check failed! Expected 0.0, got {val}"
print("[OK] SMAPE metric sanity check passed.")

In [ ]:
# Execute Run 1 pipeline
from amlc.pipeline.run import run_experiment

config_file = REPO_PATH / "configs" / "run001_lgbm_tfidf.yaml"
results = run_experiment(config_file)

In [ ]:
# Display per-bucket SMAPE & Top-30 Feature Importances
import pandas as pd
from IPython.display import display

run_dir = Path(results["run_dir"])

print("=== Per-Decile SMAPE Breakdown ===")
df_bucket = pd.read_csv(run_dir / "smape_by_bucket.csv")
display(df_bucket)

print("\n=== Top 30 Feature Importances (Gain) ===")
df_imp = pd.read_csv(run_dir / "feature_importance.csv")
display(df_imp.head(30))

In [ ]:
# Diagnostics plotting: Actual vs Predicted & Residual Histogram
import matplotlib.pyplot as plt
import numpy as np

val_preds = results["val_preds"]
y_true = val_preds["price_true"].values
y_pred = val_preds["price_pred_calibrated"].values

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Log-log scatter plot
axes[0].scatter(y_true, y_pred, alpha=0.15, s=10, color="indigo")
axes[0].plot([y_true.min(), y_true.max()], [y_true.min(), y_true.max()], "r--", label="Perfect Alignment")
axes[0].set_xscale("log")
axes[0].set_yscale("log")
axes[0].set_xlabel("Actual Price (USD)")
axes[0].set_ylabel("Predicted Price (USD)")
axes[0].set_title("Validation Actual vs Predicted (Log-Log)")
axes[0].grid(True, which="both", ls="--", alpha=0.5)
axes[0].legend()

# Residual histogram
log_residuals = np.log1p(y_pred) - np.log1p(y_true)
axes[1].hist(log_residuals, bins=50, color="teal", edgecolor="black", alpha=0.7)
axes[1].set_xlabel("Log Residual (log1p(y_pred) - log1p(y_true))")
axes[1].set_ylabel("Count")
axes[1].set_title("Validation Residual Distribution")
axes[1].grid(True, ls="--", alpha=0.5)

plt.tight_layout()
plt.show()

In [ ]:
# Zip run artifacts for easy local download and repository sync
import shutil
zip_output_path = f"/kaggle/working/{results['run_id']}_results"
shutil.make_archive(zip_output_path, 'zip', results['run_dir'])
print(f"[OK] Created downloadable run zip archive: {zip_output_path}.zip")

## Observations

*(Record findings, error slice patterns, and insights here after executing on Kaggle)*